# 05 · Copper price-adjustment clause

Which copper clause should the next contract carry? Four designs, backtested on the World Bank Pink Sheet copper series from 2016:

| Design | Rule |
|---|---|
| Fixed annual | price set each January on December's copper, fixed for the year |
| Monthly pass-through | price reset every month on the previous month's copper |
| Quarterly, 3-month lag | price reset each quarter on the previous quarter's average |
| ±5% band | price reset only when copper moves more than 5% from the last reset |

The contract price is the should-cost at the index value. The supplier buys copper at spot, so its margin shows who carries the risk. **Buyer metric:** mean drift of the price from the January price (budget certainty). **Supplier metric:** spread of its margin. Code in `src/clause.py`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src import clause
from src import should_cost as sc
from src.db import DEFAULT_DB, connect

con = connect(DEFAULT_DB)
copper = con.execute("SELECT month, copper_usd_t FROM copper_monthly ORDER BY month").df()
copper = copper.set_index(pd.to_datetime(copper["month"]))["copper_usd_t"]
copper.loc["2015-10-01":].plot(figsize=(9, 3), title="Copper, USD/t (Pink Sheet)");

## Body harness made in Brazil

In [ ]:
body = clause.backtest(copper, sc.BY_FAMILY["harness_body"], "BR", start="2016-01-01")
body.metrics.assign(design=body.metrics["design"].map(clause.LABELS)).round(4)

In [ ]:
clause.recommend(body.metrics)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
for d in clause.DESIGNS:
    axes[0].plot(body.prices[d], label=clause.LABELS[d])
    axes[1].plot(body.margins[d], label=clause.LABELS[d], lw=0.9)
axes[0].set(ylabel="Contract price, USD", title="Contract price")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set(ylabel="Supplier margin", title="Supplier margin, monthly")
axes[0].legend(fontsize=8, ncol=2, frameon=False)
fig.tight_layout()

In [ ]:
yearly = body.margins.groupby(body.margins.index.year).mean().rename(columns=clause.LABELS)
yearly.round(3)

## Does the answer change with the part or the country?

The same backtest for a body harness made in China and for a signal connector made in Brazil.

In [ ]:
cases = {
    "body harness, BR": ("harness_body", "BR"),
    "body harness, CN": ("harness_body", "CN"),
    "HV battery harness, BR": ("harness_hv_battery", "BR"),
    "signal connector, BR": ("connector_signal", "BR"),
}
rows = []
for name, (family, country) in cases.items():
    r = clause.backtest(copper, sc.BY_FAMILY[family], country, start="2016-01-01")
    rows.append({"case": name, **clause.recommend(r.metrics)})
pd.DataFrame(rows)

## Reading

- **Fixed annual** is the buyer's best case: no budget drift inside the year and the lowest total paid (1.1% below monthly pass-through over 2016–2026, because copper trended up and the price always lagged it). It also pushes all copper risk onto the supplier: its yearly margin fell below 3% in 2021 and 2026 and went negative in December 2025. A supplier exposed like that pads its quotes or walks away.
- **Monthly pass-through** is the supplier's best case: its margin barely moves, but the price changes every month (128 times in ten years).
- **Quarterly adjustment with a 3-month lag** is the compromise under the decision rule (the design whose weaker side is least weak): budget drift of 1.7%, a margin spread of 1.9 points and a worst month still above 2%.
- The **±5% band** protects the supplier almost as well as monthly pass-through with only 33 changes in ten years, at slightly more budget drift than the quarterly clause. It is the fallback if the supplier insists on margin protection.
- The ranking is the same for a Chinese-made harness, an HV battery harness and a signal connector, so one clause design can serve the whole category.

**Recommendation:** open with fixed annual, expect monthly from the supplier, and settle on quarterly adjustment on the previous quarter's average copper.